## Analisis exploratorio de RAW dataset LA/FT 

In [44]:
# Librerias

import pandas as pd


In [45]:
# lectura y creacion de dataframe

df_laft = pd.read_csv('./data/bronze/dataset_riesgo_laft.csv', sep=";", encoding="utf-8")
df_laft.head()

,transaction_id,customer_id,timestamp,amount_USD,currency_original,merchant_or_receiver,payment_description,customer_country,customer_activity,channel
0,TRX_000000,879ea,2025-01-01 00:04:00,95.28,COP,REC_3507,Suscripción mensual,CO,Asalariado,WEB
1,TRX_000001,7a39a,2025-01-01 00:12:00,86.74,COP,REC_9574,Compra supermercado,CO,Asalariado,WEB
2,TRX_000002,edf73,2025-01-01 00:46:00,33.26,COP,REC_2226,Transferencia familiar,CO,Comerciante,BRANCH
3,TRX_000003,b73d6,2025-01-01 01:10:00,89.10,COP,REC_1854,Transferencia familiar,CO,Asalariado,BRANCH
4,TRX_000004,8c19b,2025-01-01 01:25:00,13.65,COP,REC_2354,Compra supermercado,CO,Comerciante,WEB


In [46]:
# Descripcion del dataframe
df_laft.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12121 entries, 0 to 12120
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   transaction_id        12121 non-null  object 
 1   customer_id           12121 non-null  object 
 2   timestamp             12121 non-null  object 
 3   amount_USD            12121 non-null  float64
 4   currency_original     12121 non-null  object 
 5   merchant_or_receiver  12121 non-null  object 
 6   payment_description   12060 non-null  object 
 7   customer_country      12121 non-null  object 
 8   customer_activity     12121 non-null  object 
 9   channel               12121 non-null  object 
dtypes: float64(1), object(9)
memory usage: 947.1+ KB


### 1. Transacciones

Existen 12.121 trasacciones unicas en el dataframe

In [47]:
df_laft['transaction_id'].nunique()

12121

### 2. Customers
El dataframe contiene 600 clientes unicos

In [48]:
df_laft['customer_id'].nunique()

600

### 3. Timestamp
La variable que captura la fecha de la trasaccion contiene 12.121 registros, con 11.552 registros unicos inicia en '2025-01-01 00:04:00' y finaliza en '2025-03-30 23:59:00'

In [49]:
df_laft['timestamp'].describe()

count                   12121
unique                  11552
top       2025-01-26 17:02:00
freq                        3
Name: timestamp, dtype: object

In [50]:
df_laft['timestamp'].min()

'2025-01-01 00:04:00'

In [51]:
df_laft['timestamp'].max()

'2025-03-30 23:59:00'

In [52]:
# Verificar si existen filas duplicadas idénticas en todas las columnas
duplicados_totales = df_laft.duplicated().sum()
print(f"Filas completamente duplicadas: {duplicados_totales}")

# Verificar si hay clientes realizando más de una transacción en el MISMO segundo exacto
mismo_cliente_mismo_tiempo = df_laft.duplicated(subset=['customer_id', 'timestamp']).sum()
print(f"Mismo cliente transaccionando en el mismo segundo exacto: {mismo_cliente_mismo_tiempo}")

Filas completamente duplicadas: 0
Mismo cliente transaccionando en el mismo segundo exacto: 5


### 4. Amount_USD

In [53]:
df_laft['amount_USD'].describe()

count    12121.000000
mean        89.065998
std        290.112109
min          5.500000
25%         36.300000
50%         56.500000
75%         86.840000
max       7374.900000
Name: amount_USD, dtype: float64

### 5. Currency original

In [54]:
df_laft['currency_original'].unique()

array(['COP', 'MXN', 'USD'], dtype=object)

In [55]:
df_laft.groupby('currency_original').agg(
    Cantidad_tx=('transaction_id', 'count'),
    Clientes_unicos=('customer_id', 'nunique'),
    Merchants_unicos=('merchant_or_receiver', 'nunique'),
    Total_USD=('amount_USD', 'sum'),
    Monto_Promedio_USD=('amount_USD', 'mean'),
    Monto_Mediana_USD=('amount_USD', 'median'),
    Monto_Max_USD=('amount_USD', 'max'),
    # Ticket promedio por cliente
    USD_por_Cliente=('amount_USD', lambda x: x.sum() / x.nunique())
).reset_index()

,currency_original,Cantidad_tx,Clientes_unicos,Merchants_unicos,Total_USD,Monto_Promedio_USD,Monto_Mediana_USD,Monto_Max_USD,USD_por_Cliente
0,COP,5875,311,4322,475021.16,80.854666,54.660,3880.13,103.198166
1,MXN,6150,301,4450,439741.15,71.502626,57.540,1422.60,91.612740
2,USD,96,34,31,164806.65,1716.735937,351.745,7374.90,1734.806842


### 6. Merchant or Receiver

In [56]:
df_laft['merchant_or_receiver'].nunique()

6634

### 8. Customer country

In [57]:
df_laft['customer_country'].unique()

array(['CO', 'MX', 'MEX', 'COL', 'PA', 'KY', 'VG'], dtype=object)

In [58]:
df_laft.groupby('customer_country').agg(
    Cantidad_tx=('transaction_id', 'count'),
    Clientes_unicos=('customer_id', 'nunique'),
    Merchants_unicos=('merchant_or_receiver', 'nunique'),
    Total_USD=('amount_USD', 'sum'),
    Monto_Promedio_USD=('amount_USD', 'mean'),
    Monto_Mediana_USD=('amount_USD', 'median'),
    Monto_Max_USD=('amount_USD', 'max'),
    # Ticket promedio por cliente
    USD_por_Cliente=('amount_USD', lambda x: x.sum() / x.nunique())
).reset_index()

,customer_country,Cantidad_tx,Clientes_unicos,Merchants_unicos,Total_USD,Monto_Promedio_USD,Monto_Mediana_USD,Monto_Max_USD,USD_por_Cliente
0,CO,5844,312,4290,480021.36,82.139179,54.780,3880.13,104.443290
1,COL,58,53,58,3912.77,67.461552,57.055,207.95,67.461552
2,KY,13,13,13,50474.82,3882.678462,3767.380,7170.67,3882.678462
3,MEX,62,57,61,4079.42,65.797097,58.425,400.54,65.797097
4,MX,6101,305,4415,439842.36,72.093486,57.610,1422.60,92.210138
5,PA,36,15,11,66655.04,1851.528889,320.790,7374.90,1904.429714
6,VG,7,7,7,34583.19,4940.455714,4876.050,7171.49,4940.455714


### 9. Customer activity

In [59]:
df_laft['customer_activity'].unique()

array(['Asalariado', 'Comerciante', 'Independiente', 'Consultor Int.'],
      dtype=object)

In [60]:
df_laft.groupby('customer_activity').agg(
    Cantidad_tx=('transaction_id', 'count'),
    Clientes_unicos=('customer_id', 'nunique'),
    Merchants_unicos=('merchant_or_receiver', 'nunique'),
    Total_USD=('amount_USD', 'sum'),
    Monto_Promedio_USD=('amount_USD', 'mean'),
    Monto_Mediana_USD=('amount_USD', 'median'),
    Monto_Max_USD=('amount_USD', 'max'),
    # Ticket promedio por cliente
    USD_por_Cliente=('amount_USD', lambda x: x.sum() / x.nunique())
).reset_index()

,customer_activity,Cantidad_tx,Clientes_unicos,Merchants_unicos,Total_USD,Monto_Promedio_USD,Monto_Mediana_USD,Monto_Max_USD,USD_por_Cliente
0,Asalariado,3997,198,3217,274338.91,68.636205,55.970,1681.68,81.285603
1,Comerciante,4084,210,3252,302328.58,74.027566,56.075,2148.36,87.302506
2,Consultor Int.,30,29,30,143671.77,4789.059000,4625.000,7374.90,4789.059000
3,Independiente,4010,218,3209,359229.70,89.583466,57.490,3880.13,105.811399


### 10. Channel

In [61]:
df_laft['channel'].unique()

array(['WEB', 'BRANCH', 'APP'], dtype=object)

In [62]:
df_laft.groupby('channel').agg(
    Cantidad_tx=('transaction_id', 'count'),
    Clientes_unicos=('customer_id', 'nunique'),
    Merchants_unicos=('merchant_or_receiver', 'nunique'),
    Total_USD=('amount_USD', 'sum'),
    Monto_Promedio_USD=('amount_USD', 'mean'),
    Monto_Mediana_USD=('amount_USD', 'median'),
    Monto_Max_USD=('amount_USD', 'max'),
    # Ticket promedio por cliente
    USD_por_Cliente=('amount_USD', lambda x: x.sum() / x.nunique())
).reset_index()

,channel,Cantidad_tx,Clientes_unicos,Merchants_unicos,Total_USD,Monto_Promedio_USD,Monto_Mediana_USD,Monto_Max_USD,USD_por_Cliente
0,APP,3833,194,3075,283713.61,74.018682,58.53,1422.60,85.817789
1,BRANCH,4009,200,3293,276444.91,68.956076,53.27,2241.24,81.909603
2,WEB,4279,249,3371,519410.44,121.385941,57.53,7374.90,144.080566


---

## Data Cleaning

In [63]:
# Copia de trabajo
df_laft_clean = df_laft.copy()

# Conversion de timestamp a Datetime
df_laft_clean['timestamp'] = pd.to_datetime(df_laft_clean['timestamp'])

# Conversion de identificadores y texto libre a String
string_cols = ['transaction_id', 'customer_id', 'merchant_or_receiver']
for col in string_cols:
    df_laft_clean[col] = df_laft_clean[col].astype("string")

# Tratar el texto libre para NLP (reemplazar nulos con cadena vacía y de tipo a string)
df_laft_clean['payment_description'] = (
    df_laft_clean['payment_description']
    .fillna("")
    .astype("string")
)

# Conversion de variables cualitativas a Categorical
cat_cols = ['currency_original', 'customer_country', 'customer_activity', 'channel']
for col in cat_cols:
    df_laft_clean[col] = df_laft_clean[col].astype("category")

#      Verificación de resultado y ahorro de memoria
print("=== NUEVA ESTRUCTURA DE TIPOS ===")
df_laft_clean.info()

=== NUEVA ESTRUCTURA DE TIPOS ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12121 entries, 0 to 12120
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   transaction_id        12121 non-null  string        
 1   customer_id           12121 non-null  string        
 2   timestamp             12121 non-null  datetime64[ns]
 3   amount_USD            12121 non-null  float64       
 4   currency_original     12121 non-null  category      
 5   merchant_or_receiver  12121 non-null  string        
 6   payment_description   12121 non-null  string        
 7   customer_country      12121 non-null  category      
 8   customer_activity     12121 non-null  category      
 9   channel               12121 non-null  category      
dtypes: category(4), datetime64[ns](1), float64(1), string(4)
memory usage: 616.5 KB


### 8. Customer country

In [64]:
# Diccionario de estandarización (Mapear ISO-3 a ISO-2)
country_map = {
    'COL': 'CO',
    'MEX': 'MX'
}

# Aplicar el reemplazo
df_laft_clean['customer_country'] = df_laft_clean['customer_country'].replace(country_map)

# Si la columna era de tipo 'category', reasegurar la categoría
df_laft_clean['customer_country'] = df_laft_clean['customer_country'].astype('category')

C:\Users\Hugo\AppData\Local\Temp\ipykernel_14284\735394371.py:8: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_laft_clean['customer_country'] = df_laft_clean['customer_country'].replace(country_map)


In [65]:
df_laft['customer_country'].unique()

array(['CO', 'MX', 'MEX', 'COL', 'PA', 'KY', 'VG'], dtype=object)

In [66]:
df_laft_clean.groupby('customer_country').agg(
    Cantidad_tx=('transaction_id', 'count'),
    Clientes_unicos=('customer_id', 'nunique'),
    Merchants_unicos=('merchant_or_receiver', 'nunique'),
    Total_USD=('amount_USD', 'sum'),
    Monto_Promedio_USD=('amount_USD', 'mean'),
    Monto_Mediana_USD=('amount_USD', 'median'),
    Monto_Max_USD=('amount_USD', 'max'),
    # Ticket promedio por cliente
    USD_por_Cliente=('amount_USD', lambda x: x.sum() / x.nunique())
).reset_index()

C:\Users\Hugo\AppData\Local\Temp\ipykernel_14284\11047605.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_laft_clean.groupby('customer_country').agg(


,customer_country,Cantidad_tx,Clientes_unicos,Merchants_unicos,Total_USD,Monto_Promedio_USD,Monto_Mediana_USD,Monto_Max_USD,USD_por_Cliente
0,CO,5902,312,4323,483934.13,81.994939,54.79,3880.13,104.521410
1,KY,13,13,13,50474.82,3882.678462,3767.38,7170.67,3882.678462
2,MX,6163,305,4451,443921.78,72.030144,57.61,1422.60,92.233904
3,PA,36,15,11,66655.04,1851.528889,320.79,7374.90,1904.429714
4,VG,7,7,7,34583.19,4940.455714,4876.05,7171.49,4940.455714


In [67]:
df_laft_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12121 entries, 0 to 12120
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   transaction_id        12121 non-null  string        
 1   customer_id           12121 non-null  string        
 2   timestamp             12121 non-null  datetime64[ns]
 3   amount_USD            12121 non-null  float64       
 4   currency_original     12121 non-null  category      
 5   merchant_or_receiver  12121 non-null  string        
 6   payment_description   12121 non-null  string        
 7   customer_country      12121 non-null  category      
 8   customer_activity     12121 non-null  category      
 9   channel               12121 non-null  category      
dtypes: category(4), datetime64[ns](1), float64(1), string(4)
memory usage: 616.3 KB


In [68]:
# Guardado del dataframe limpio
df_laft_clean.to_csv("./data/silver/dataset_riesgo_laft_silver.csv", sep=",", index=False, encoding="utf-8")